# KAN-NNUE Training in Bullet

Train KAN (B-spline) and baseline (SCReLU) NNUE networks on the same data,
then compare loss curves.

**Runtime**: Use a GPU runtime (T4 or better). Go to Runtime > Change runtime type > GPU.

Architecture:
- **KAN**: `768 -> ft(128) -> KAN(256->128) -> KAN(128->1) -> sigmoid`
- **Baseline**: `768 -> ft(128) -> SCReLU -> 256->128 -> SCReLU -> 128->1 -> sigmoid`

## 1. Setup: Install Rust & Clone Repo

In [16]:
%%bash
# Install Rust (if not already installed)
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    echo 'source $HOME/.cargo/env' >> ~/.bashrc
fi
source $HOME/.cargo/env
rustc --version
cargo --version


  stable-x86_64-unknown-linux-gnu unchanged - rustc 1.94.1 (e408947bf 2026-03-25)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.94.1 (e408947bf 2026-03-25)
cargo 1.94.1 (29ea6fb6a 2026-03-24)


info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
warn: Updating existing toolchain, profile choice will be ignored
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: default toolchain set to stable-x86_64-unknown-linux-gnu


In [17]:
%%bash
source $HOME/.cargo/env

# Clone the bullet fork with KAN support
if [ ! -d /content/bullet ]; then
    git clone https://github.com/y0sif/bullet.git /content/bullet
fi
cd /content/bullet
git pull origin main
git log --oneline -5

Already up to date.
c03b243 Fix training speed: match kanue epoch size (488 batches = ~8M positions)
5f448d4 Fix Colab notebook: install zstd, drop wget --show-progress
6f26c6f Add CUDA backend for BSplineBasis + Colab training notebook
81848c5 Update training examples: SfBinpackLoader, baseline comparison, CPU device fix
1a024fe Add unit tests for BSplineBasis CPU implementation


From https://github.com/y0sif/bullet
 * branch            main       -> FETCH_HEAD


## 2. Download Training Data

In [18]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test77.binpack ]; then
    echo "Downloading test77 binpack from HuggingFace (~1.3 GB compressed)..."
    wget -q -O test77.binpack.zst \
        "https://huggingface.co/datasets/linrock/test77/resolve/main/test77-2022-01-jan-2tb7p.binpack.zst"
    echo "Download complete. Decompressing..."
    zstd -d test77.binpack.zst -o test77.binpack --rm
    echo "Done!"
fi

ls -lh test77.binpack

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
-rw-r--r-- 1 root root 2.7G Apr 10 18:22 test77.binpack


## 3. Build Both Examples

In [19]:
%%bash
source $HOME/.cargo/env
cd /content/bullet

# Check CUDA availability
nvidia-smi || echo "WARNING: No GPU detected. Training will be very slow."
echo "---"
nvcc --version || echo "WARNING: nvcc not found. Will try CPU fallback."

Fri Apr 10 18:37:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
%%bash
source $HOME/.cargo/env
cd /content/bullet

# Build both examples in release mode
# Uses default features (cuda) if nvcc is available, falls back to cpu
if command -v nvcc &> /dev/null; then
    echo "Building with CUDA support..."
    cargo build --release --example kan_simple --example kan_baseline 2>&1
else
    echo "Building with CPU only (no CUDA toolkit found)..."
    cargo build --no-default-features --features cpu --release --example kan_simple --example kan_baseline 2>&1
fi
echo "Build complete!"

Building with CUDA support...
    Finished `release` profile [optimized] target(s) in 0.05s
Build complete!


## 4. Train Baseline (SCReLU)

In [21]:
import subprocess, sys, os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

cmd = ["cargo", "run", "--release", "--example", "kan_baseline"]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd="/content/bullet", text=True, bufsize=1)
log = open("/content/baseline_log.txt", "w")
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
    log.write(line)
log.close()
proc.wait()
print(f"\nExited with code {proc.returncode}")

    Finished `release` profile [optimized] target(s) in 0.05s
     Running `target/release/examples/kan_baseline`
Training Preamble
Net Name               : kan-baseline
Batch Size             : 16384
Batches / Superbatch   : 488
Positions / Superbatch : 7995392
Start Superbatch       : 1
End Superbatch         : 40
Eval Scale             : 400
Save Rate              : 10
WDL Scheduler          : constant 0.75
LR Scheduler           : start 0.001 gamma 0.1 drop every 18 superbatches
Threads                : 4
Output Path            : checkpoints
Beginning Training
superbatch 1 [0.0% (0/488 batches, 66219 pos/sec)]
Estimated time to end of superbatch: infs     superbatch 1 [26.2% (128/488 batches, 1564586 pos/sec)]
Estimated time to end of superbatch: 3.8s     superbatch 1 [52.5% (256/488 batches, 1787567 pos/sec)]
Estimated time to end of superbatch: 2.1s     superbatch 1 [78.7% (384/488 batches, 1902290 pos/sec)]
Estimated time to end of superbatch: 0.9s     superbatch 1 | time 4.2s |

## 5. Train KAN

In [22]:
import subprocess, sys, os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

cmd = ["cargo", "run", "--release", "--example", "kan_simple"]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd="/content/bullet", text=True, bufsize=1)
log = open("/content/kan_log.txt", "w")
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
    log.write(line)
log.close()
proc.wait()
print(f"\nExited with code {proc.returncode}")

    Finished `release` profile [optimized] target(s) in 0.11s
     Running `target/release/examples/kan_simple`
Training Preamble
Net Name               : kan-simple
Batch Size             : 16384
Batches / Superbatch   : 488
Positions / Superbatch : 7995392
Start Superbatch       : 1
End Superbatch         : 40
Eval Scale             : 400
Save Rate              : 10
WDL Scheduler          : constant 0.75
LR Scheduler           : start 0.001 gamma 0.1 drop every 18 superbatches
Threads                : 4
Output Path            : checkpoints
Beginning Training
superbatch 1 [0.0% (0/488 batches, 98633 pos/sec)]
Estimated time to end of superbatch: infs     superbatch 1 [26.2% (128/488 batches, 543035 pos/sec)]
Estimated time to end of superbatch: 10.9s     superbatch 1 [52.5% (256/488 batches, 554512 pos/sec)]
Estimated time to end of superbatch: 6.9s     superbatch 1 [78.7% (384/488 batches, 549950 pos/sec)]
Estimated time to end of superbatch: 3.1s     superbatch 1 | time 14.5s | runn

## 6. Compare Loss Curves

In [ ]:
import re
import matplotlib.pyplot as plt

def strip_ansi(s):
    """Remove ANSI escape codes from string."""
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

def parse_bullet_log(path):
    """Parse Bullet training log to extract superbatch loss values."""
    losses = []
    with open(path) as f:
        for line in f:
            clean = strip_ansi(line)
            # Bullet format: "superbatch N | time Ts | running loss X.XXXXXX | ..."
            m = re.search(r'superbatch\s+(\d+)\s+\|.*?running loss\s+([\d.]+)', clean)
            if m:
                losses.append((int(m.group(1)), float(m.group(2))))
    return losses

baseline = parse_bullet_log('/content/baseline_log.txt')
kan = parse_bullet_log('/content/kan_log.txt')

if baseline and kan:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot([x[0] for x in baseline], [x[1] for x in baseline], label='Baseline (SCReLU)', linewidth=2)
    ax.plot([x[0] for x in kan], [x[1] for x in kan], label='KAN (B-spline)', linewidth=2)
    ax.set_xlabel('Superbatch')
    ax.set_ylabel('Loss')
    ax.set_title('KAN vs Baseline NNUE Training Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/kan_vs_baseline.png', dpi=150)
    plt.show()

    baseline_final = baseline[-1][1]
    kan_final = kan[-1][1]
    improvement = (baseline_final - kan_final) / baseline_final * 100
    print(f'\nBaseline final loss: {baseline_final:.6f}')
    print(f'KAN final loss:      {kan_final:.6f}')
    print(f'Improvement:         {improvement:+.1f}%')
else:
    print('Could not parse training logs.')
    if not baseline:
        print('  - baseline_log.txt: no superbatch loss lines found')
        with open('/content/baseline_log.txt') as f:
            print('  First 5 lines:', [strip_ansi(l.strip()) for l in f.readlines()[:5]])
    if not kan:
        print('  - kan_log.txt: no superbatch loss lines found')
        with open('/content/kan_log.txt') as f:
            print('  First 5 lines:', [strip_ansi(l.strip()) for l in f.readlines()[:5]])

## 7. Save Results

Checkpoints are saved to `/content/bullet/checkpoints/`. Copy to Drive if needed:

In [ ]:
import shutil, os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/kanue-bullet')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

for src in ['/content/kan_log.txt', '/content/baseline_log.txt', '/content/kan_vs_baseline.png']:
    if os.path.exists(src):
        shutil.copy2(src, DRIVE_BASE / os.path.basename(src))
        print(f'Copied {src}')
    else:
        print(f'Skipped {src} (not found)')

# Copy checkpoints if they exist
ckpt_dir = Path('/content/bullet/checkpoints')
if ckpt_dir.exists():
    dest = DRIVE_BASE / 'checkpoints'
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(ckpt_dir, dest)
    print(f'Copied checkpoints -> {dest}')